# Notebook 1 - Why You Need a High-Water Mark

## The story (in plain words)

Imagine a small bank with **one main branch (the leader)** and **two backup branches (followers)** that copy every transaction the main branch records. Each transaction is appended to a notebook called the **log**.

A customer asks the main branch: *"Did my transfer go through?"* The main branch peeks at the latest line in its own notebook and says **yes**. The customer leaves happy.

Then a fire destroys the main branch before the backups copied that last line. A backup is promoted to be the new "main" - but the transaction the customer was told about **is gone**. That is a *lost write*, and it is exactly what we want to prevent.

The trick is simple: **do not let clients see a write until enough copies of it exist**. The boundary between "safely replicated" and "still tentative" is called the **high-water mark (HWM)**.

In this notebook we first show the **bad version**: the leader exposes its newest entry immediately, no matter how many followers have it. We will watch a write get lost.


## Setup

```bash
cd 02-distributed-primitives/high-water-mark
uv sync
```

In VS Code, pick the `.venv` kernel (top-right of the notebook). If it does not appear, reload the window: `Cmd+Shift+P` -> **Reload Window**.


## Building blocks

A **node** is just any participant in the cluster. Each node owns a **log** - an append-only list of entries (think: every entry is a transaction).

We model nodes with a tiny dataclass - no networking, no async, no databases. The whole point is to see the algorithm clearly.


In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Node:
    name: str
    log: List[str] = field(default_factory=list)

    def __repr__(self):
        return f"Node({self.name}, log={self.log})"


## BAD: leader exposes its tail before followers ack

In the *naive* design, the leader appends a new entry and **immediately** tells clients about it. Replication to followers happens later (or sometimes never, if the leader dies first).


In [ ]:
leader = Node("leader")
followers = [Node("f1"), Node("f2")]

def naive_replicate(entry, deliver_to):
    """Leader appends locally and ships to *some* followers.

    In real life 'deliver_to' is decided by the network: maybe a packet
    is dropped, maybe a follower is slow. Here we just pass the list to
    make the failure mode obvious.
    """
    leader.log.append(entry)
    for f in deliver_to:
        f.log.append(entry)

# Three writes; the third never reaches any follower.
naive_replicate("A", followers)        # everyone has A
naive_replicate("B", followers[:1])    # only f1 has B
naive_replicate("C", [])               # only the leader has C

# Client reads the leader's tail and acts on it (e.g. confirms an order).
client_saw = leader.log[-1]
print("client read:", client_saw)
print("leader log :", leader.log)
print("f1 log     :", followers[0].log)
print("f2 log     :", followers[1].log)


## The leader crashes - pick a new one

A common election rule is *"the follower with the longest log wins"*. Sounds reasonable. Let us see what happens to the entry the client was told about.


In [ ]:
new_leader = max(followers, key=lambda f: len(f.log))
print("new leader   :", new_leader.name)
print("new log      :", new_leader.log)
print()
print("client saw   :", client_saw)
print("lost write?  :", client_saw not in new_leader.log)

# Assert the loss, so nobody can "fix" this notebook by accident and leave
# notebook 2 explaining a problem that no longer reproduces.
assert client_saw == "C"
assert client_saw not in new_leader.log, "the lost-write bug did not reproduce"
assert new_leader.log == ["A", "B"], new_leader.log
print("\n💥 the client was told 'C' succeeded; after failover 'C' does not exist anywhere")


## What just happened?

The client got an answer based on data that lived **only on the leader**. When the leader vanished, that data vanished too - but the client still believes it is there. That is a silent **data loss**.

| What clients can read | Risk |
|---|---|
| Only the leader's tail | Whatever the leader uniquely holds is lost on crash |
| Only entries that a quorum has | A new leader will have those entries, so committed reads survive |

The second row is the **high-water mark** rule. We implement it in the next notebook.

> **Real-world parallel.** Early MongoDB defaulted to `w:1` (acknowledge after the primary writes). Operators learned the hard way during failovers that some "successful" writes had silently disappeared, which is why production deployments now use `w:majority`. Same problem, same fix.
